In [ ]:
#使用inspect.exe中查看的信息定位ui控件
from pyweixin import Navigator
main_window=Navigator.open_weixin()
#消息编辑框
edit=main_window.child_window(control_type='Edit',auto_id='chat_input_field',class_name='mmui::ChatInputField')
if edit.exists(timeout=0.1):
    edit.type_keys('你好')
else:
    main_window.close()
    print(f'请先打开切换当前界面至某一个好友的聊天框！')

In [ ]:
#获取ui控件的属性
from pyweixin import Navigator
main_window=Navigator.open_weixin()
print(f'微信主窗口的名字是:{main_window.window_text()}')
print(f'微信主窗口的类名是:{main_window.class_name()}')


In [ ]:
#Application连接窗口
from pywinauto import Application
'''
pywinauto Application类可以用来连接到正在运行的应用:
使用connect方法时有四个可选参数,分别是:
path:应用程序.exe路径
handle:窗口句柄(inspect中可查看)
pid:应用的process id(可在任务管理器中右键查看详细信息中看到)
timeout:连接超时时间
...
...
以及其他定位参数:比如title,control_type,auto_id,class_name
'''

pid=22736
handle=0x309A2
path=r"D:\Weixin\Weixin.exe"
wechat_app1=Application(backend='uia').connect(process=pid)
wechat_app2=Application(backend='uia').connect(handle=handle)
wechat_app3=Application(backend='uia').connect(path=path)
#不建议使用定位ui控件的方式连接
#wechat_app4=Application(backend='uia').connect(title='微信',class_name='mmui::MainWindow')
'''在连接到应用后便可以使用.window方法来连接当前应用程序的主窗口了
连接窗口时和我们定位ui控件一样,传入的参数有title,control_type,auto_id,class_name
等当前窗口的ui属性,当然还支持直接传入handle(最稳定),这里建议能够获取到窗口句柄就尽量用窗口句柄
特别是对于4.1版本微信,使用常规的title,control_type,auto_id,class_name等ui属性无法直接定位到窗口
'''
main_window=wechat_app3.window(handle=handle)
#main_window=wechat_app3.window(title='微信',class_name='mmui::MainWindow')
if main_window.exists(timeout=1):
    main_window.restore()

In [ ]:
#Desktop连接窗口
from pywinauto import Desktop
desktop=Desktop(backend='uia')
'''直接使用desktop.window方法来连接当前应用程序的主窗口
连接窗口时和我们定位ui控件一样,传入的参数有title,control_type,auto_id,class_name
等当前窗口的ui属性,当然还支持直接传入handle(最稳定),这里建议能够获取到窗口句柄就尽量用窗口句柄
特别是对于4.1版本微信,使用常规的title,control_type,auto_id,class_name等ui属性无法直接定位到窗口
'''
handle=0x309A2
main_window=desktop.window(handle=handle)
#main_window=desktop.window(title='微信',class_name='mmui::MainWindow')
if main_window.exists(timeout=1):
    main_window.restore()

In [ ]:
#连接到4.1+微信窗口示例
import re
import win32gui,win32con,win32api,win32com.client
from pywinauto import Desktop,WindowSpecification
desktop=Desktop(backend='uia')
OffLineButton={'title':'当前网络不可用','control_type':'Button'}#网络不好时,微信顶部的按钮

class NotStartError(Exception):
    def __init__(self, Error='微信未启动,请先启动并登录微信后再使用pyweixin!'):
        super().__init__(Error)
class NotLoginError(Exception):
    def __init__(self, Error='微信未登录,请先点击登录后再使用pyweixin!'):
        super().__init__(Error)  
class NetWorkError(Exception):
    def __init__(self, Error='当前网络不可用,无法进行UI自动化!'):
        super().__init__(Error)
class NotFoundError(Exception):
    def __init__(self,Error='无法识别定位到微信主界面,请在微信登录前运行无障碍服务(讲述人)后再尝试!'):
        super().__init__(Error)

def is_weixin_running()->bool:
    '''
    该方法通过检测当前windows系统的进程中
    是否有Weixin.exe该项进程来判断微信是否在运行
    '''
    wmi=win32com.client.GetObject('winmgmts:')
    processes=wmi.InstancesOf('Win32_Process')
    for process in processes:
        if process.Name.lower()=='Weixin.exe'.lower():
            return True
    return False

class WxWindowManage():
    '''用来查找活跃的微信窗口,实例化后使用find_wx_window方法'''
    def __init__(self):
        self.hwnd=0
        self.possible_windows=[]
        self.window_type=1#1为主界面,0为登录界面
        self.classname_pattern=re.compile(r'Qt\d+QWindowIcon')#Qt51514QWindowIcon,QT窗口通用的classname

    def filter(self,hwnd,param):
        #EnumDesktopWindows的回调函数
        classname=win32gui.GetClassName(hwnd)
        if self.classname_pattern.match(classname):
            self.possible_windows.append(hwnd) 
    
    def find_wx_window(self):
        '''当微信在运行时,即使关闭掉窗口。win32gui也可以找到窗口句柄
        不过win32gui获取到的classname是通用的qt窗口类名
        pywinauto可以获取到真正的窗口类名mmui::,其中mm与微信移动版的package包名一致
        猜测是微信为了全平台的一致性''' 
        
        win32gui.EnumDesktopWindows(0,self.filter,None)  
        self.possible_windows=[hwnd for hwnd in self.possible_windows 
        if 'mmui::MainWindow' in desktop.window(handle=hwnd).class_name() or 'mmui::LoginWindow' in desktop.window(handle=hwnd).class_name()]
        if self.possible_windows:
            self.hwnd=self.possible_windows[0]
            if desktop.window(handle=self.hwnd).class_name()=='mmui::LoginWindow':
                self.window_type=0#登录界面
        return self.hwnd


def open_weixin(is_maximize:bool=False,window_size:tuple=(1000,1000))->WindowSpecification:
    '''
    打开微信(微信需要提前登录)
    Args:
        is_maximize:微信界面是否全屏,默认不全屏
        window_size:微信主界面大小,默认(1000,100),可GlobalConfig.window_size=(width,height)全局控制
    '''
    def move_window_to_center(window:WindowSpecification,is_maximize:bool):
        #将微信主界面移动到窗口正中间,并调整全屏
        window.restore()
        win32gui.SetWindowPos(window.handle,win32con.HWND_TOPMOST, 
        0, 0,window_size[0],window_size[1],win32con.SWP_NOMOVE)
        window_width,window_height=window_size[0],window_size[1]
        screen_width,screen_height=win32api.GetSystemMetrics(win32con.SM_CXSCREEN),win32api.GetSystemMetrics(win32con.SM_CYSCREEN)
        new_left=(screen_width-window_width)//2
        new_top=(screen_height-window_height)//2
        if screen_width!=window_width:
            #移动窗口到屏幕中央
            win32gui.MoveWindow(window.handle,new_left,new_top,window_width,window_height,True)
        ###############################
        if is_maximize:
            win32gui.SendMessage(window.handle, win32con.WM_SYSCOMMAND, win32con.SC_MAXIMIZE,0)
        if not is_maximize:
            win32gui.SendMessage(window.handle, win32con.WM_SYSCOMMAND, win32con.SC_RESTORE,0)
        return window
    wx=WxWindowManage()
    is_running=is_weixin_running()
    if not is_running:#微信不在运行,主界面看不到窗口，需要先启动
        raise NotStartError
    handle=wx.find_wx_window()
    if handle==0:
        raise NotFoundError
    #只有在窗口激活的时候
    if wx.window_type==0:#微信在运行,但是是登录界面
        raise NotLoginError
    if wx.window_type==1:#微信在运行，主界面存在(可能被关闭或者可见)
        wx_window=desktop.window(handle=handle)
        main_window=move_window_to_center(wx_window,is_maximize=is_maximize)
        win32gui.SetWindowPos(main_window.handle,win32con.HWND_NOTOPMOST,
        0, 0, 0, 0,win32con.SWP_NOMOVE|win32con.SWP_NOSIZE)
    offline_button=main_window.child_window(**OffLineButton)
    if offline_button.exists(timeout=0.1):
        main_window.close()
        raise NetWorkError('当前网络不可用,无法进行UI自动化!')
    return main_window
main_window=open_weixin()
weixin_button=main_window.child_window(title="微信",control_type="Button")

In [ ]:
#pywinatuo backend为win32 自动操作记事本
from pywinauto.application import Application
app=Application(backend="win32").start("notepad.exe")
app=app.connect(class_name='Notepad',timeout=5)
main_window=app.window(class_name='Notepad')
main_window.child_window(class_name="RichEditD2DPT").set_text('pywinauto自动化操作记事本')

In [ ]:
#.child_window方法定位控件示例
from pyweixin import Navigator
'''亦可直接使用Navigator.open_moments()
这里为.child_window定位控件的示例
'''
#pywinauto定位控件的方法参数都是kwargs,我们可以预先将控件的信息写成字典形式，然后使用**解包传入
#可以看做POM(PageObjectModel)封装
Toolbar={'title':'导航','control_type':'ToolBar'}#主界面左侧的侧边栏
Moments={'title':'朋友圈','control_type':'Button','class_name':"mmui::XTabBarItem"}#主界面左侧的朋友圈按钮
main_window=Navigator.open_weixin(is_maximize=False)
#先定位到左侧工具栏
toolbar=main_window.child_window(**Toolbar)
# #然后再左侧工具栏中定位朋友圈按钮
moments_button=toolbar.child_window(**Moments)
'''亦可省去中间变量,直接链式查找'''
#main_window.child_window(**Toolbar).child_window(**Moments).click_input()
'''或者直接在main_window内查找'''
#moments_button=main_window.child_window(**Moments)
moments_button.click_input()#最后点击朋友圈按钮

In [ ]:
#.child_window方法定位层级较深的控件示例
from pyweixin import Navigator
#pywinauto定位控件的方法参数都是kwargs,我们可以预先将控件的信息写成字典形式，然后使用**解包传入
#可以看做POM(PageObjectModel)封装
Weixin={'title':'微信','control_type':'Button','class_name':"mmui::XTabBarItem"}
SessionList={'title':'会话','control_type':'List','framework_id':'Qt'}
main_window=Navigator.open_weixin()
weixin_button=main_window.window(**Weixin)
weixin_button.click_input()
session_list=main_window.window(**SessionList)
print(session_list.dump_tree())
for listitem in session_list.children():
    print(listitem.automation_id().replace('session_item_',''))

In [ ]:
#.window方法定位控件示例
from pyweixin import Navigator
'''亦可直接使用Navigator.open_moments()
这里为.window定位控件的示例
'''
#pywinauto定位控件的方法参数都是kwargs,我们可以预先将控件的信息写成字典形式，然后使用**解包传入
#可以看做POM(PageObjectModel)封装
Toolbar={'title':'导航','control_type':'ToolBar'}#主界面左侧的侧边栏
Moments={'title':'朋友圈','control_type':'Button','class_name':"mmui::XTabBarItem"}#主界面左侧的朋友圈按钮
main_window=Navigator.open_weixin(is_maximize=False)
'''
.window与.child_window相比,没有什么区别,window()方法已被标记为弃用,它是child_window()方法的别名
'''
#先定位到左侧工具栏
toolbar=main_window.window(**Toolbar)
# #然后再左侧工具栏中定位朋友圈按钮
moments_button=toolbar.window(**Moments)
'''亦可省去中间变量,直接链式查找'''
#main_window.window(**Toolbar).window(**Moments).click_input()
'''或者直接在main_window内查找'''
#moments_button=main_window.window(**Moments)
moments_button.click_input()#最后点击朋友圈按钮

In [ ]:
#descendants方法查找控件
import time
from pyweixin import Navigator
'''使用descendants方法查找控件时,传入的参数与child_window,window一样
返回值为所有符合条件的ui控件列表,如果你坚信只有一个控件，那么使用列表索引来获取该控件
如果没有符合条件的ui控件那么该方法返回的结果是[]
所以使用descendants时我们需要根据结果是否为空来判断控件是否存在
'''
main_window=Navigator.open_weixin()
#pywinauto定位控件的方法参数都是kwargs,我们可以预先将控件的信息写成字典形式，然后使用**解包传入
#可以看做POM(PageObjectModel)封装
search={'title':'搜索','control_type':'Edit','class_name':"mmui::XValidatorTextEdit"}
search_bar1=main_window.child_window(**search)
search_bar2=main_window.descendants(**search)
print(search_bar1.exists(timeout=1))#微信顶部的搜索栏使用child_window方法还定位不到,只能使用descendants,而且使用dump_tree也看不到
if search_bar2:
    search_bar2[0].click_input()
    search_bar2[0].set_text(f'文件传输助手')
    time.sleep(1)
    search_bar2[0].type_keys('{ENTER}')


In [ ]:
#.children()查找控件
from pyweixin import Navigator
'''.children()主要用来获取当前控件下的子控件,不传入任何参数时,获取的是所有子控件列表
如果传入了参数,那么边会按照传入的参数筛选查找,最终返回值与descendants方法一样都是一个控件列表
与descendants方法唯一不同的地方在于descendants支持深度查找,而.children只能获取当期控件下的子控件
'''
#pywinauto定位控件的方法参数都是kwargs,我们可以预先将控件的信息写成字典形式，然后使用**解包传入
#可以看做POM(PageObjectModel)封装
Weixin={'title':'微信','control_type':'Button','class_name':"mmui::XTabBarItem"}
SessionList={'title':'会话','control_type':'List','framework_id':'Qt'}
main_window=Navigator.open_weixin()
weixin_button=main_window.window(**Weixin)
weixin_button.click_input()
session_list=main_window.window(**SessionList)
for child in session_list.children(control_type='ListItem'):
    print(child.automation_id())

In [ ]:
#exists方法判断控件是否存在
from pyweixin import Navigator
'''
exists方法参数:
timeout:超时时间,超过这个时间不再判断直接返回False
retry_interval:timeout范围内每次重试的间隔,timeout为1,retry_interval为0.1,就是1s内重试10次
重试的过程就是一直在调用底层的查找逻辑
返回值:
exists返回值:
exists方法的返回值为bool,即使超时也不会产生异常,适合在开发代码中直接使用
'''
friend='文件传输助手'
main_window=Navigator.open_weixin()
chat_item=main_window.child_window(control_type='ListItem',auto_id=f'session_item_{friend}')
if chat_item.exists(timeout=1,retry_interval=0.1):#切换聊天界面之friend
    chat_item.click_input()
else:
    print(f'未在当前可见会话列表内看到好友 {friend}')

In [ ]:
#wait等待控件出现
import time
from pywinauto.timings import TimeoutError
from pyweixin import Navigator
'''
wait方法用来等待控件出现,超时后会引发TimeoutError,这个时候需要异常处理,
一般不建议用,如果只是单纯的判断是否存在建议使用exists
参数:
wait_for:等待控件出现的条件,包括:
    'exists' means that the window is a valid handle
    'visible' means that the window is not hidden
    'enabled' means that the window is not disabled
    'ready' means that the window is visible and enabled
    'active' means that the window is active
timeout:超时时间,超过这个时间不再判断直接返回False
retry_interval:timeout范围内每次重试的间隔,timeout为1,retry_interval为0.1,就是1s内重试10次
'''
main_window=Navigator.open_weixin()
#pywinauto定位控件的方法参数都是kwargs,我们可以预先将控件的信息写成字典形式，然后使用**解包传入
#可以看做POM(PageObjectModel)封装
search={'title':'搜索','control_type':'Edit','class_name':"mmui::XValidatorTextEdit"}
search_bar2=main_window.descendants(**search)
try:
    search_bar1=main_window.child_window(**search).wait(timeout=1,wait_for='ready')#微信顶部的搜索栏使用child_window方法还定位不到,只能使用descendants,而且使用dump_tree也看不到
except TimeoutError:#超时会引起
    print(f'已超时捕获TimeoutError异常,该控件可能不存在,使用descendants方法尝试!')
    if search_bar2:
        search_bar2[0].click_input()
        search_bar2[0].set_text(f'文件传输助手')
        time.sleep(1)
        search_bar2[0].type_keys('{ENTER}')


In [ ]:
#wait_not等待控件不出现
import time
from pywinauto.timings import TimeoutError
from pyweixin import Navigator
'''
wait_not方法用来等待控件不出现,超时后会引发TimeoutError,这个时候需要异常处理,
一般不建议用,如果只是单纯的判断是否存在建议使用exists

参数:
wait_for_not:等待控件不出现的条件,包括:
    'exists' means that the window is a valid handle
    'visible' means that the window is not hidden
    'enabled' means that the window is not disabled
    'ready' means that the window is visible and enabled
    'active' means that the window is active
timeout:超时时间,超过这个时间不再判断直接返回False
retry_interval:timeout范围内每次重试的间隔,timeout为1,retry_interval为0.1,就是1s内重试10次
'''
main_window=Navigator.open_weixin()
#pywinauto定位控件的方法参数都是kwargs,我们可以预先将控件的信息写成字典形式，然后使用**解包传入
#可以看做POM(PageObjectModel)封装
search={'title':'搜索','control_type':'Edit','class_name':"mmui::XValidatorTextEdit"}
search_bar2=main_window.descendants(**search)
try:
    search_bar1=main_window.child_window(**search).wait_not(timeout=1,wait_for_not='ready')#微信顶部的搜索栏使用child_window方法还定位不到,只能使用descendants,而且使用dump_tree也看不到
    print(f'该控件的确不存在,使用descendants方法定位')
    search_bar2[0].click_input()
    search_bar2[0].set_text(f'文件传输助手')
    time.sleep(1)
    search_bar2[0].type_keys('{ENTER}')
except TimeoutError:#超时会引起
    print(f'已超时捕获TimeoutError异常,该控件的确存在!')


In [ ]:
#print_control_identifiers()
from pyweixin import Navigator
'''print_control_identifiers()主要用来输出当前控件下的结构
'''
#pywinauto定位控件的方法参数都是kwargs,我们可以预先将控件的信息写成字典形式，然后使用**解包传入
#可以看做POM(PageObjectModel)封装
Weixin={'title':'微信','control_type':'Button','class_name':"mmui::XTabBarItem"}
SessionList={'title':'会话','control_type':'List','framework_id':'Qt'}
main_window=Navigator.open_weixin()
weixin_button=main_window.window(**Weixin)
weixin_button.click_input()
session_list=main_window.window(**SessionList)
print(session_list.print_control_identifiers())

In [ ]:
#pywinauto获取控件基本属性
from pyweixin import Navigator
'''
基本属性:可以直接通过.访问到的属性，大致有以下这些
'''
main_window=Navigator.open_weixin()
print(f'微信主窗口的名字是:{main_window.window_text()}')
print(f'微信主窗口的类名是:{main_window.class_name()}')
print(f'微信主窗口的AutomationId是:{main_window.automation_id()}')
print(f'微信主窗口的控件ID是:{main_window.control_id()}')
print(f'微信主窗口的窗口句柄是:{main_window.handle}')
print(f'微信主窗口的进程ID是:{main_window.process_id()}')
print(f'微信主窗口的BoundingRectangle是:{main_window.rectangle()}')
print(f'微信主窗口的父控件是:{main_window.parent()}') 
print(f'微信主窗口的子控件是:{main_window.children()}') 

In [ ]:
#pywinauto访问底层元素信息
from pyweixin import Navigator
'''
底层属性:需要通过element_info获取的一些属性,与基本属性有重合
基本属性中的那些方法,大部分都是在类内做了个getter接口,直接返回这里的值
'''
main_window=Navigator.open_weixin()
element=main_window.element_info
print(f'微信主窗口的类名是:{element.class_name}')
print(f'微信主窗口的类型是:{element.control_type}')
print(f'微信主窗口的名字是:{element.name}')
print(f'微信主窗口的AutomationID是:{element.automation_id}') 
print(f'微信主窗口的句柄是:{element.handle}') 
print(f'微信主窗口的进程ID是:{element.process_id}') 
print(f'微信主窗口的runtime_id是:{element.runtime_id}') 
print(f'微信主窗口的框架ID是:{element.framework_id}') 
print(f'微信主窗口的子控件是:{element.children()}') 
print(f'微信主窗口的父控件是:{element.parent}')         

In [ ]:
#pywinauto ui控件的状态属性
from pyweixin import Navigator
main_window=Navigator.open_weixin()
NotificationButton={'title':'通知','control_type':'Button'}
SessionList={'title':'会话','control_type':'List','framework_id':'Qt'}
newMessageAlertCheckBox={'title':'新消息通知声音','control_type':'CheckBox'}
session_list=main_window.window(**SessionList)
session_list.type_keys('{HOME}')
print(f'会话列表是否启用:{session_list.is_enabled()}')
print(f'会话列表是否可见:{session_list.is_visible()}')
print(f'会话列表是否有键盘聚焦:{session_list.has_keyboard_focus()}')
for listitem in session_list.children():
    if listitem.is_selected():
        print(f'当前会话列表中被选中的聊天对象是:{listitem.automation_id().replace('session_item_','')}')
settings_window=Navigator.open_settings()
notification_button=settings_window.child_window(**NotificationButton)
notification_button.click_input()
newMessage_checkbox=settings_window.child_window(**newMessageAlertCheckBox)
print(f'新消息通知声音复选框是否被选中:{newMessage_checkbox.get_toggle_state()}')
settings_window.close()
main_window.close()

In [ ]:
#pywinauto ui控件的文本和值
from pyweixin import Navigator
profile_pane,main_window=Navigator.open_friend_profile(friend='小号测试')
text_uis=profile_pane.descendants(control_type='Text')
texts=[item.window_text() for item in text_uis]#使用window_text()或texts()[0]获取文本
texts=[item.texts()[0] for item in text_uis]
main_window.close()
print(texts)

In [ ]:
# pywinauto对控件通用的点击操作
from pywinauto import base_wrapper#ctrl+鼠标左键单击跳转后即可查看下列方法源代码
from pyweixin import Navigator
CurrentChatEdit={'control_type':'Edit','auto_id':'chat_input_field'}#微信主界面下当前的聊天窗口
main_window=Navigator.open_dialog_window(friend='文件传输助手')
edit_area=main_window.child_window(**CurrentChatEdit)
'''pywinauto中,直接作用在控件上的点击操作主要包含以下五种:
click_input():单击
double_click_input():双击
right_click_input():右键单击
press_mouse_input():按下鼠标
release_mouse_input():释放鼠标
'''
edit_area.click_input()   
edit_area.double_click_input() 
edit_area.right_click_input() 
edit_area.press_mouse_input()  
edit_area.release_mouse_input() 

In [ ]:
# pywinauto对控件通用的输入操作
from pywinauto.controls import uia_controls#ctrl+鼠标单击跳转即可看到下列方法的源代码
from pyweixin import Navigator
CurrentChatEdit={'control_type':'Edit','auto_id':'chat_input_field'}#微信主界面下当前的聊天窗口
main_window=Navigator.open_dialog_window(friend='文件传输助手')
edit_area=main_window.child_window(**CurrentChatEdit)
'''pywinauto中,直接作用在控件上的点击操作主要包含以下两种:
edit_area.type_keys("text",with_spaces=True): 打字的方式向编辑框内输入文本        
edit_area.type_keys("{ENTER}"):按下键盘上的Enter键
edit_area.set_text("text"):直接设置编辑框的文本(会清除掉先前的内容)
'''
edit_area.type_keys("你 好,文 件 传 输 助 手！",with_spaces=True)       
edit_area.type_keys("{ENTER}")#按下回车
edit_area.set_text("你 好,文 件 传 输 助 手！")#直接设置文本，不打字
edit_area.type_keys("{ENTER}")#按下回车

In [ ]:
#pywinauto鼠标相关操作
from pywinauto import mouse
'''click参数:
button:'left'或'right',点击的是鼠标左键还是右键
coords:屏幕坐标(x,y),注意x,y均为正整数
'''
mouse.click(button='left',coords=(200,200))#运行后点击指定的位置
#当然,理论上还可以使用for循环实现连点操作
for _ in range(2):
    mouse.click(button='left',coords=(200,200))

In [ ]:
#pywinauto鼠标相关操作
from pywinauto import mouse
'''mouse.scroll参数:
coords:屏幕坐标(x,y),注意x,y均为正整数
wheel_dist:鼠标滚动幅度，需要是整数,为负时鼠标滑论向后滚动,页面向下滚动,为正时相反
'''
mouse.scroll(wheel_dist=-20,coords=(1000,500))#在指定位置滚动

In [ ]:
import subprocess
import sounddevice as sd
import soundfile as sf
from pyweixin import Navigator,GlobalConfig
SendAudioButon={'title_re':'发语音','control_type':'Button'} 
#转换编码格式
def convert_m4a_to_wav(input_file,output_file,channels=1):
    '''使用ffmpeg转换手机录音M4A到WAV'''
    audio,sample_rate=sf.read(input_file)
    command=[
        "ffmpeg",
        "-i", input_file,         
        "-ar", str(sample_rate),   
        "-ac", str(channels),
        '-y',   
        output_file
    ]
    subprocess.run(command,check=True,capture_output=True)
    return input_file

def get_default_output()->int:
    output_devives=sd.default.device#可能有多个
    default_output=output_devives[0]#默认第一个
    for device in output_devives:
        query=sd.query_devices(device)
        if query.get('name') is not None and 'cable input' in query.get('name').lower():
            default_output=device
    return default_output

def play_audio(audio_file:str):
    audio,samplerate=sf.read(audio_file)
    sd.default.device=get_default_output()
    sd.play(audio,samplerate)
    sd.wait()

def send_audio(friend,audio_file,search_pages:int=None,is_maximize:bool=None,close_weixin:bool=None):
    if is_maximize is None:
        is_maximize=GlobalConfig.is_maximize
    if close_weixin is None:
        close_weixin=GlobalConfig.close_weixin
    if search_pages is None:
        search_pages=GlobalConfig.search_pages
    version=GlobalConfig.version
    if '4.1.9' in version:
        main_window=Navigator.open_dialog_window(friend=friend,search_pages=search_pages,is_maximize=is_maximize)
        send_audio_button=main_window.child_window(**SendAudioButon)